_**Import PKGs**_

In [ ]:
import gmso 
import random
import numpy as np
import mbuild as mb
from gmso.core.forcefield import ForceField
from gmso.external import from_mbuild
from gmso.parameterization import apply
from mbuild.lib.recipes.polymer import Polymer
from rdkit.Chem import Draw
from IPython.display import display

In [ ]:
import io
import warnings
import fresnel
import IPython
import packaging.version
import PIL
import math


In [ ]:
from gmso.utils.nx_utils import plot_networkx_atomtypes
from gmso.formats.lammpsdata import write_lammpsdata

_**Build the Structure of Repeat Unit**_

In [ ]:
comp = mb.load('CC=C(C)C', smiles=True) 
#comp.visualize()

_**Check the Order of Atoms**_

In [ ]:
comp_rdkmol=comp.to_rdkit()
img=Draw.MolToImage(comp_rdkmol)
display(img)

_**Build the Structure of Single chain**_

In [ ]:
chain = Polymer()

chain.add_monomer(compound=comp,
                  indices=[5, 12],
                  separation=.154,
                  replace=True)
chain.add_end_groups(mb.load('CC=C(C)C',smiles=True), 
                     index=12,
                     separation=0.154, label="head", duplicate=False)

chain.add_end_groups(mb.load('CC=C(C)C',smiles=True), 
                     index=5,
                     separation=0.154, label="tail", duplicate=False)
chain.build(n=8, sequence='A')
chain.visualize(show_ports=True)


_**Build the Structure of System Containing Several Chains**_

In [ ]:
# the pattern we generate puts points in the xy-plane, so we'll rotate the polymer
# so that it is oriented normal to the xy-plane
chain.rotate(np.pi/2, [0, 1, 0])

# define a compound to hold all the polymers
system = mb.Compound()

# create a pattern of points to fill a disk
# patterns are generated between 0 and 1,
# and thus need to be scaled to provide appropriate spacing
pattern_disk = mb.DiskPattern(49)
pattern_disk.scale(5)

# now clone the polymer and move it to the points in the pattern
for pos in pattern_disk:
    current_polymer = mb.clone(chain)
    current_polymer.translate(pos)
    system.add(current_polymer)

system.visualize()


In [ ]:
system.save('system.pdb')

_**Apply the Force Field to the System**_

In [ ]:
#Without ignoring any types, we will find the improper types are missing.
gmso_oplsaa = ForceField("oplsaa1.0.1.xml")
gmso_topology = from_mbuild(system)
apply(top=gmso_topology, forcefields=gmso_oplsaa, identify_connections=True, ignore_params=[])


_**Check the missing types**_

In [ ]:
#ignore the missing types
gmso_oplsaa = ForceField("oplsaa1.0.1.xml")
gmso_topology = from_mbuild(comp)
apply(top=gmso_topology, forcefields=gmso_oplsaa, identify_connections=True, ignore_params=["impropers"], remove_untyped=False)

In [ ]:
#check improper types
for improper in gmso_topology.impropers:
    print(improper.improper_type)

In [ ]:
#Here we can see all impropers those are missing, including CT-HC-HC-HC
for improper in gmso_topology.impropers:
    if improper.improper_type is None:
        atom_types = [a.atom_type.name for a in improper.connection_members]
        atom_classes = [a.atom_type.atomclass for a in improper.connection_members]
        print(atom_types)
        print(atom_classes)
        print()

_**Check the TYPE of Atoms in a picture**_

In [ ]:
plot_networkx_atomtypes(gmso_topology)

_**WRITE LAMMPS DATA**_

In [ ]:
write_lammpsdata(gmso_topology, filename='parmed_top.lammps', atom_style='full')